In [ ]:
from src.slm_initialise import slm_initialise
from src.utils import *
from src.plots import plot_fit,bb,g
from slmsuite.holography.algorithms import FeedbackHologram
from slmsuite.holography.toolbox import convert_vector
from slmsuite.holography.toolbox.phase import zernike_sum,zernike_get_string,ZERNIKE_NAMES 
from src.algos import *
from src.zernike import get_zernike_calibrations,get_cholo_phase,get_cholo_bb_g_phase,fit_metric,fit_metric_udf,plot_parabola,get_zernike_dict_from_metrics_df,get_zernike_metrics_df,zernike_names
from src import global_store
%load_ext autoreload
%autoreload 2

In [ ]:
fs = slm_initialise(force=True,verbose=True)
calibrate(fs,vendor = False,refresh = False)
sp_calibrate(fs,refresh=False)
plot_cam(fs,phase="zero",centre=True, save=True)

In [ ]:
# ! pip install scikit-optimize

# Bayesian opt for bottle beam

In [ ]:
import numpy as np
np.random.seed(237)
import matplotlib.pyplot as plt
from skopt.plots import plot_gaussian_process

In [ ]:
target_shape = fs.cam.shape
j,i = np.meshgrid(np.arange(target_shape[0]), np.arange(target_shape[1]), indexing='ij')

centrex = 1180
centrey = 230
n_spots = 5
r = 50
n = 4
rad = 100
bb_locations = np.array([[centrex+(i-2)*rad ,centrey + (j-1)*rad] for i in range(n) for j in range(n)]).T
spot_locations = np.array([[centrex+(i-2)*rad + 50,centrey + (j-1)*rad + 50] for i in range(n) for j in range(n)])



zernike_calibrations = []
nz = 10
for i in range(len(spot_locations)):
    zernike_xy = np.flip(toolbox.convert_vector([1440+10-20,1024+65+20] - spot_locations[i],from_units="ij",to_units="zernike",hardware=fs).T[0])
    zernike_arr = np.concatenate([zernike_xy, np.zeros(nz -2, dtype=zernike_xy.dtype)])
    zernike_calibrations.append({i+1: float(v) for i, v in enumerate(zernike_arr)})
zernike_indices = [i+1 for i in range(nz)]

In [ ]:
s = 9
base_phase = get_cholo_phase(zernike_calibrations=zernike_calibrations,fs = fs) + bottle_beam_phase(fs)
best = -1

In [ ]:
def f(x): 
    zernike_list = [i for i in range(3,nz+1)]
    # zernike_list = [4]
    phase = base_phase + zernike_sum(fs,tuple(zernike_list),tuple(x),use_mask = False)
    image = get_cam(fs=fs,phase = phase)
    patch = image[spot_locations[s][1]-50:spot_locations[s][1]+50,spot_locations[s][0]-50:spot_locations[s][0]+50]
    fitres = fit_bottle_beam(patch, patch_size=80)
    print(fitres['r2'])
    return -fitres["r2"]

In [ ]:
from skopt import gp_minimize
from skopt.learning import GaussianProcessRegressor
from skopt.learning.gaussian_process.kernels import Matern, WhiteKernel,RationalQuadratic
res = gp_minimize(f,                  # the function to minimize
                  [(-1.0, 1.0) for i in range(nz-2)],      # the bounds on each dimension of x
                  acq_func="LCB",      # the acquisition function
                  base_estimator = GaussianProcessRegressor(kernel=1**2 * RationalQuadratic(length_scale=1.0, alpha=0.1) + WhiteKernel(noise_level=0.01),
                        n_restarts_optimizer=2, noise=0.1,
                        normalize_y=True, random_state=822569775),
                  x0 = [0.0 for i in range(nz-2)],
                  n_calls=500,         # the number of evaluations of f
                  n_initial_points=1,  # the number of random initialization points
                  noise=0.1,       # the noise level (optional)
                  random_state=1234)

In [ ]:
from skopt.plots import plot_convergence
plot_convergence(res);

In [ ]:
# from pathlib import Path
# from skopt import dump, load

# save_dir = Path(base_path) / "data" / "zernike" / "raw"
# save_dir.mkdir(parents=True, exist_ok=True)
# save_path = save_dir / "gp_500_25z.skopt"

# # save without the objective function
# dump(res, save_path, store_objective=False)

# # load later
# res3 = load(save_path)

# # (optional) resume optimization later
# # res2 has x_iters and func_vals:
# # res_cont = gp_minimize(f, space, x0=res2.x_iters, y0=res2.func_vals, n_calls=50)


In [ ]:
plt.scatter(np.array(res['x_iters']).T[6],res['func_vals'])

In [ ]:
plt.imshow(image)

# bottle beam fits 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.ndimage import center_of_mass, map_coordinates

# bottle-beam model over an (x,y) grid
def bbn(xy, A, w, xc, yc, B):
    x, y = xy
    r2 = (x - xc)**2 + (y - yc)**2
    return B + A * (r2**2) * np.exp(-r2 / (w**2))  # r^4 = (r^2)^2

def fit_bottle_beam(
    img,
    top_percentile_for_peak=99.5,
    n_ring_samples=720,
    n_line_samples=1024,
    make_plots=True
):
    img = np.asarray(img, dtype=float)
    H, W = img.shape
    yy, xx = np.meshgrid(np.arange(H), np.arange(W), indexing="ij")  # rows, cols

    # ---- initial guesses ----
    B0 = np.median(img)
    thresh = np.percentile(img, top_percentile_for_peak)
    weights = np.clip(img - B0, 0, None) * (img >= thresh)
    if weights.sum() == 0:
        weights = np.clip(img - B0, 0, None) + 1e-12
    yc0, xc0 = H/2,W/2
    yc0 = np.nan_to_num(yc0, nan=H/2)
    xc0 = np.nan_to_num(xc0, nan=W/2)

    # coarse radial profile to locate initial peak radius
    r = np.sqrt((xx - xc0)**2 + (yy - yc0)**2)
    r_int = r.astype(int)
    r_max_possible = int(np.ceil(r.max()))
    sums   = np.bincount(r_int.ravel(), weights=img.ravel(), minlength=r_max_possible+1)
    counts = np.bincount(r_int.ravel(), minlength=r_max_possible+1)
    radial_profile = sums / np.maximum(counts, 1)
    r_peak0 = int(np.nanargmax(radial_profile))
    w0 = max(r_peak0 / np.sqrt(2), 1.0)
    I_peak = radial_profile[r_peak0]
    A0 = max((I_peak - B0) / ( (r_peak0**4) * np.exp(-(r_peak0**2)/(w0**2)) + 1e-12 ), 1e-6)
    p0 = [A0, w0, xc0, yc0, B0]

    # sane bounds
    lower = [0.0,  1e-3,         0.0,     0.0,     np.min(img)-abs(np.ptp(img))]
    upper = [np.inf, max(H, W),  W-1+0.5, H-1+0.5, np.max(img)+abs(np.ptp(img))]

    # ---- fit ----
    print(p0)
    popt, pcov = curve_fit(
        bbn,
        (xx.ravel(), yy.ravel()),
        img.ravel(),
        p0=p0, bounds=(lower, upper), maxfev=20000
    )
    A, w, xc, yc, B = popt
    r_peak = np.sqrt(2) * w

    if make_plots:
        # 1) image with center + peak ring
        fig1, ax1 = plt.subplots(figsize=(5.5, 5))
        im = ax1.imshow(img, origin="upper", aspect="equal")
        ax1.scatter([xc], [yc], s=60, edgecolor="white", facecolor="none", linewidth=1.5, label="Center")
        th = np.linspace(0, 2*np.pi, 400)
        ax1.plot(xc + r_peak*np.cos(th), yc + r_peak*np.sin(th), 'w--', lw=1.2, label="Peak ring")
        ax1.set_title("Fit: center & peak ring")
        ax1.set_xlim(0, W-1); ax1.set_ylim(H-1, 0)
        ax1.legend(loc="lower right"); fig1.colorbar(im, ax=ax1, shrink=0.85)
        plt.tight_layout()

        # 2) radial profile (data vs model)
        r_bins = np.arange(0, int(np.ceil(r.max())) + 1).astype(float)
        model_profile = B + A * (r_bins**4) * np.exp(-(r_bins**2)/(w**2))
        fig2, ax2 = plt.subplots(figsize=(6.5, 4))
        ax2.plot(r_bins, radial_profile[:len(r_bins)], label="Data (azimuthal avg)")
        ax2.plot(r_bins, model_profile, '--', label="Model fit")
        ax2.axvline(r_peak, linestyle=':', lw=1, label=f"r_peak ≈ {r_peak:.2f}")
        ax2.set_xlabel("Radius (px)"); ax2.set_ylabel("Intensity"); ax2.set_title("Radial profile")
        ax2.legend(); ax2.grid(True, alpha=0.3); plt.tight_layout()

        # 3) ring scan at r = r_peak
        theta = np.linspace(0, 2*np.pi, n_ring_samples, endpoint=False)
        x_ring = xc + r_peak * np.cos(theta)
        y_ring = yc + r_peak * np.sin(theta)
        ring_vals = map_coordinates(img, [y_ring, x_ring], order=1, mode="nearest")
        fig3, ax3 = plt.subplots(figsize=(7.5, 3.5))
        ax3.plot(np.degrees(theta), ring_vals)
        ax3.set_xlim(0, 360)
        ax3.set_xlabel("Angle (deg)"); ax3.set_ylabel("Intensity")
        ax3.set_title("Ring scan at peak radius")
        ax3.grid(True, alpha=0.3)
        mu, std = np.mean(ring_vals), np.std(ring_vals)
        ax3.text(0.01, 0.95, f"mean={mu:.3g}, std={std:.3g}, CV={std/max(mu,1e-12):.2%}",
                 transform=ax3.transAxes, va="top")
        plt.tight_layout()

        # 4) X-scan through center (y = yc)
        x_line = np.linspace(0, W-1, n_line_samples)
        y_line = np.full_like(x_line, yc, dtype=float)
        x_scan = map_coordinates(img, [y_line, x_line], order=1, mode="nearest")
        # model along x: r^2 = (x-xc)^2
        dx = x_line - xc
        x_model = B + A * (dx**4) * np.exp(-(dx**2)/(w**2))
        fig4, ax4 = plt.subplots(figsize=(7.5, 3.5))
        ax4.plot(x_line, x_scan, label="Data")
        ax4.plot(x_line, x_model, '--', label="Model")
        ax4.set_xlabel("x (px)"); ax4.set_ylabel("Intensity")
        ax4.set_title("X-scan through fitted center")
        ax4.grid(True, alpha=0.3); ax4.legend(); plt.tight_layout()

        # 5) Y-scan through center (x = xc)
        y_line2 = np.linspace(0, H-1, n_line_samples)
        x_line2 = np.full_like(y_line2, xc, dtype=float)
        y_scan = map_coordinates(img, [y_line2, x_line2], order=1, mode="nearest")
        dy = y_line2 - yc
        y_model = B + A * (dy**4) * np.exp(-(dy**2)/(w**2))
        fig5, ax5 = plt.subplots(figsize=(7.5, 3.5))
        ax5.plot(y_line2, y_scan, label="Data")
        ax5.plot(y_line2, y_model, '--', label="Model")
        ax5.set_xlabel("y (px)"); ax5.set_ylabel("Intensity")
        ax5.set_title("Y-scan through fitted center")
        ax5.grid(True, alpha=0.3); ax5.legend(); plt.tight_layout()

    return dict(A=A, w=w, xc=xc, yc=yc, B=B, r_peak=r_peak)

# usage:
# params = fit_bottle_beam(img, make_plots=True)
# print(params)


In [ ]:
params = fit_bottle_beam(patch, make_plots=True)
print(params)